In [ ]:
%pip install numba -q
%pip install engineering-notation -q
%pip install opencv-contrib-python -q
%pip install argcomplete -q
%pip install dv-processing -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 83.5 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/SensorsINI/v2e
%cd /content/v2e
#!python setup.py develop
%pip install .
!mkdir input
%cd /content/
%pwd

Cloning into 'v2e'...
remote: Enumerating objects: 3227, done.
remote: Counting objects: 100% (1201/1201), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 3227 (delta 1156), reused 1102 (delta 1102), pack-reused 2026 (from 1)
Receiving objects: 100% (3227/3227), 34.25 MiB | 14.15 MiB/s, done.
Resolving deltas: 100% (2385/2385), done.
/content/v2e
Processing /content/v2e
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 8.7 MB/s eta 0:00:00
  Created wheel for v2e: filename=v2e-1.5.1-py3-none-any.whl size=116998 sha256=a44f4ef2750dfe4c208b86c912190ffa702ad09e42697c91c34dca2787b2ab5a
  Stored in directory: /tmp/pip-ephem-wheel-cache-cd4wyhd_/wheels/f4/f3/51/80837a929df608ceb06a5ecdb5f272f60a25522cfd8be8913d
Successfully built v2e
/content


'/content'

In [ ]:
import gdown
import os
from tqdm.notebook import tqdm
import shutil
import cv2
import pandas as pd
import numpy as np
from PIL import Image
import h5py
import pandas as pd
import subprocess


os.makedirs('/content/v2e/input/', exist_ok=True)

url = 'https://drive.google.com/uc?id=1ETID_4xqLpRBrRo1aOT7Yphs3QqWR_fx'
output = '/content/v2e/input/SuperSloMo39.ckpt'
gdown.download(url, output, quiet=False)


Downloading...
From (original): https://drive.google.com/uc?id=1ETID_4xqLpRBrRo1aOT7Yphs3QqWR_fx
From (redirected): https://drive.google.com/uc?id=1ETID_4xqLpRBrRo1aOT7Yphs3QqWR_fx&confirm=t&uuid=d4d7cc8a-97bf-46d9-995e-2709a728998c
To: /content/v2e/input/SuperSloMo39.ckpt
100%|██████████| 158M/158M [00:07<00:00, 20.0MB/s]


'/content/v2e/input/SuperSloMo39.ckpt'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Single video

In [ ]:
seq = 'bolt_1s'
initials = 'vlra'

In [ ]:
os.makedirs('/content/video', exist_ok=True)
shutil.copy(f'/content/drive/MyDrive/Diploma_project/own_videos/{initials}/28.03.2026/mp4/{seq}.mp4', '/content/video')

In [ ]:
#@title In the file explorer, right click the video file, and then click "Copy Path". Then, paste the path here: { run: "auto", vertical-output: true }
video_path = f"/content/video/{seq}.mp4" #@param {type:"string"}

import os
if video_path != "" and os.path.isfile(video_path):
    display("The chosen video file: {}".format(video_path))
else:
    print("The file path is empty or invalid, choose a file")

In [ ]:
# play the sample video, you can ignore it, support both AVI and MP4
# you can ignore this if you don't need
if ".avi" in video_path:
    !ffmpeg -y -i $video_path /tmp/output.mp4 -loglevel quiet
else:
    !cp $video_path /tmp/output.mp4

from IPython.display import HTML
from base64 import b64encode
mp4 = open("/tmp/output.mp4",'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("<video controls src={} width=50%/>".format(data_url))

In [ ]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError(f"Cannot open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration_sec = frame_count / fps if fps > 0 else 0

print(f"FPS: {fps}")
print(f"Frame count: {frame_count}")
print(f"Duration: {duration_sec:.2f} sec")

cap.release()

In [ ]:
#@title **Configure V2E Options** { run: "auto", vertical-output: true, display-mode: "both" }
#@markdown The full documentation is here: https://github.com/SensorsINI/v2e#render-emulated-dvs-events-from-conventional-video

#@markdown This GUI supports only common use configs, for finer control, please check the v2e documentation and edit the code below.

#@markdown ---
#@markdown ## Output Options

output_folder = "/content/v2e-output" #@param {type:"string"}
#@markdown - folder to store outputs. (default: v2e-output)
overwrite = True #@param {type:"boolean"}
#@markdown - overwrites files in existing folder (checks existence of non-empty output_folder). (default: True)
unique_output_folder = False #@param {type:"boolean"}
#@markdown - makes unique output folder based on output_folder, e.g. output1 (if non-empty output_folder already exists) (default: False)
out_filename = "events.h5" #@param {type:"string"}
#@markdown - Output DVS events as hdf5 event database.
davis_output = True #@param {type:"boolean"}
#@markdown - If also save frames in HDF5. (default: False)

#@markdown ### Output DVS Video Options
skip_video_output = False #@param {type:"boolean"}
#@markdown - Skip producing video outputs, including the original video, SloMo video, and DVS video (default: False)
dvs_exposure = f"duration {1/fps:.3f}" #@param {type:"string"}
#@markdown - Mode to finish DVS frame event integration: duration time: Use fixed accumulation time in seconds, e.g. --dvs_exposure duration .005; count n: Count n events per frame, -dvs_exposure count 5000; area_event N M: frame ends when any area of M x M pixels fills with N events, -dvs_exposure area_count 500 64 (default: duration 0.01)
output_mode = "dvs346" #@param ["dvs128", "dvs240", "dvs346", "dvs640", "dvs1024"]
#@markdown - This option sets the output size of the v2e. Supported models: "dvs128" (128x128), "dvs240" (240x180), "dvs346" (346x260), "dvs640" (640x480), "dvs1024" (1024x768).

#@markdown ---

#@markdown ## Input Options
input_frame_rate = 30 #@param {type:"number"}
#@markdown - Manually define the video frame rate when the video is presented as a list of image files. When the input video is a video file, this option will be ignored.
input_slowmotion_factor =  1#@param {type:"number"}
#@markdown - Sets the known slow-motion factor of the input video, i.e. how much the video is slowed down, i.e., the ratio of shooting frame rate to playback frame rate. input_slowmotion_factor<1 for sped-up video and input_slowmotion_factor>1 for slowmotion video.If an input video is shot at 120fps yet is presented as a 30fps video (has specified playback frame rate of 30Hz, according to file's FPS setting), then set --input_slowdown_factor=4.It means that each input frame represents (1/30)/4 s=(1/120)s.If input is video with intended frame intervals of 1ms that is in AVI file with default 30 FPS playback spec, then use ((1/30)s)*(1000Hz)=33.33333. (default: 1.0)

#@markdown ---

#@markdown ## DVS Time Resolution Options
disable_slomo = False #@param {type:"boolean"}
#@markdown - Disables slomo interpolation; the output DVS events will have exactly the timestamp resolution of the source video (which is perhaps modified by --input_slowmotion_factor). (default: False)
timestamp_resolution = 0.001 #@param {type:"number"}
#@markdown - (Ignored by --disable_slomo.) Desired DVS timestamp resolution in seconds; determines slow motion upsampling factor; the video will be upsampled from source fps to achieve the at least this timestamp resolution.I.e. slowdown_factor = (1/fps)/timestamp_resolution; using a high resolution e.g. of 1ms will result in slow rendering since it will force high upsampling ratio. Can be combind with --auto_timestamp_resolution to limit upsampling to a maximum limit value. (default: None)
auto_timestamp_resolution = True #@param {type:"boolean"}
#@markdown - (Ignored by --disable_slomo.) If True (default), upsampling_factor is automatically determined to limit maximum movement between frames to 1 pixel. If False, --timestamp_resolution sets the upsampling factor for input video. Can be combined with --timestamp_resolution to ensure DVS events have at most some resolution. (default: False)

# This is the SloMo model path
slomo_model = "/content/v2e/input/SuperSloMo39.ckpt"

#@markdown ---

#@markdown ## DVS Model Options
condition = "Clean" #@param ["Custom", "Clean", "Noisy"]
#@markdown - Custom: Use following slidebar to adjust your DVS model.
#@markdown - Clean: a preset DVS model, generate clean events, without non-idealities.
#@markdown - Noisy: a preset DVS model, generate noisy events.

thres = 0.2 #@param {type:"slider", min:0.05, max:1, step:0.01}
#@markdown - threshold in log_e intensity change to trigger a positive/negative event. (default: 0.2)
sigma = 0.03 #@param {type:"slider", min:0.01, max:0.25, step:0.001}
#@markdown - 1-std deviation threshold variation in log_e intensity change. (default: 0.03)
cutoff_hz = 200 #@param {type:"slider", min:0, max:300, step:1}
#@markdown - photoreceptor first-order IIR lowpass filter cutoff-off 3dB frequency in Hz - see https://ieeexplore.ieee.org/document/4444573 (default: 300)
leak_rate_hz = 5.18 #@param {type:"slider", min:0, max:100, step:0.01}
#@markdown - leak event rate per pixel in Hz - see https://ieeexplore.ieee.org/abstract/document/7962235 (default: 0.01)
shot_noise_rate_hz = 2.716 #@param {type:"slider", min:0, max:100, step:0.001}
#@markdown - Temporal noise rate of ON+OFF events in darkest parts of scene; reduced in brightest parts. (default: 0.001)

if condition == "Clean":
    thres = 0.2
    sigma = 0.02
    cutoff_hz = 0
    leak_rate_hz = 0
    shot_noise_rate_hz = 0
elif condition == "Noisy":
    thres = 0.2
    sigma_thres = 0.05
    cutoff_hz = 30
    leak_rate_hz = 0.1
    shot_noise_rate_hz = 5

#@markdown ---

v2e_command = ["v2e"]

# set the input folder
# the video_path can be a video file or a folder of images
v2e_command += ["-i", video_path]

# set the output folder
v2e_command += ["-o", output_folder]

# if the output will rewrite the previous output
if overwrite:
    v2e_command.append("--overwrite")

# if there the output folder is unique
v2e_command += ["--unique_output_folder", "{}".format(unique_output_folder).lower()]

# set output configs, for the sake of this tutorial, let's just output HDF5 record
if davis_output:
    v2e_command += ["--davis_output"]
v2e_command += ["--dvs_h5", out_filename]
v2e_command += ["--dvs_aedat2", "None"]
v2e_command += ["--dvs_text", "None"]

# in Colab, let's say no preview
v2e_command += ["--no_preview"]

# if skip video output
if skip_video_output:
    v2e_command += ["--skip_video_output"]
else:
    # set DVS video rendering params
    dvs_exposure = f"duration {1/input_frame_rate:.6f}"
    v2e_command += ["--dvs_exposure", dvs_exposure]

# set slomo related options
v2e_command += ["--input_frame_rate", "{}".format(input_frame_rate)]
v2e_command += ["--input_slowmotion_factor", "{}".format(input_slowmotion_factor)]

# set slomo data
if disable_slomo:
    v2e_command += ["--disable_slomo"]
    v2e_command += ["--auto_timestamp_resolution", "false"]
else:
    v2e_command += ["--slomo_model", slomo_model]
    if auto_timestamp_resolution:
        v2e_command += ["--auto_timestamp_resolution", "{}".format(auto_timestamp_resolution).lower()]
    else:
        v2e_command += ["--timestamp_resolution", "{}".format(timestamp_resolution)]

# threshold
v2e_command += ["--pos_thres", "{}".format(thres)]
v2e_command += ["--neg_thres", "{}".format(thres)]

# sigma
v2e_command += ["--sigma_thres", "{}".format(sigma)]

# DVS non-idealities
v2e_command += ["--cutoff_hz", "{}".format(cutoff_hz)]
v2e_command += ["--leak_rate_hz", "{}".format(leak_rate_hz)]
v2e_command += ["--shot_noise_rate_hz", "{}".format(shot_noise_rate_hz)]

# append output mode
v2e_command += [f"--{output_mode}"]

# Final v2e command

final_v2e_command = " ".join(v2e_command)

print("The Final v2e command:")
display(final_v2e_command)


In [ ]:
!$final_v2e_command

In [ ]:
#Make csv

h5_path = "/content/v2e-output/events.h5"
csv_path = "/content/v2e-output/events.csv"

with h5py.File(h5_path, "r") as f:
    events = f["events"][:]

df = pd.DataFrame(events, columns=["t", "x", "y", "p"])
df.to_csv(csv_path, index=False)

print(f"Saved: {csv_path}")

In [ ]:
video_path = "/content/v2e-output/video_orig.avi"
output_dir = "/content/v2e-output/aps"

os.makedirs(output_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise ValueError(f"Cannot open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_path = os.path.join(output_dir, f"{frame_count + 1:04d}.png")
    cv2.imwrite(frame_path, frame)
    frame_count += 1

cap.release()

print("Detected FPS:", fps)
print("Saved frames:", frame_count)
print("Output folder:", output_dir)

In [ ]:
def create_frames_from_csv(csv_path, output_dir, aps_dir, width=346, height=260):
    os.makedirs(output_dir, exist_ok=True)

    # Determine the number of frames from the APS folder
    aps_frames = sorted([
        f for f in os.listdir(aps_dir)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])
    num_frames = len(aps_frames)
    if num_frames == 0:
        raise ValueError(f"No frames found in '{aps_dir}'.")
    print(f"APS frames found: {num_frames}")

    # Read events from CSV
    df = pd.read_csv(csv_path)
    t_all = df['t'].to_numpy()
    x_all = df['x'].to_numpy()
    y_all = df['y'].to_numpy()
    p_all = df['p'].to_numpy()

    # Divide the time range evenly into num_frames intervals
    t_start = t_all[0]
    t_end   = t_all[-1]
    frame_boundaries = np.linspace(t_start, t_end, num_frames + 1)

    # Find the event end index for each frame boundary
    event_end_idxs = np.searchsorted(t_all, frame_boundaries[1:], side="right")

    start_idx = 0

    for frame_idx, end_idx in enumerate(tqdm(event_end_idxs, desc="Saving DVS frames")):
        xs = x_all[start_idx:end_idx]
        ys = y_all[start_idx:end_idx]
        ps = p_all[start_idx:end_idx]

        frame = np.zeros((height, width), dtype=np.int16)

        # Filter out events outside image bounds
        valid = (xs >= 0) & (xs < width) & (ys >= 0) & (ys < height)
        xs = xs[valid]
        ys = ys[valid]
        ps = ps[valid]

        vals = np.where(ps, 1, -1).astype(np.int16, copy=False)
        np.add.at(frame, (ys, xs), vals)

        # Visualize ON/OFF events as red/blue on a white background
        img = np.full((height, width, 3), 255, dtype=np.uint8)
        img[frame > 0] = (220, 30, 30)   # ON  → red
        img[frame < 0] = (30, 30, 200)   # OFF → blue

        # Name the output file to match the corresponding APS frame
        file_name = aps_frames[frame_idx]
        Image.fromarray(img).save(os.path.join(output_dir, file_name))

        start_idx = end_idx




# Run
create_frames_from_csv(
    csv_path="/content/v2e-output/events.csv",
    output_dir="/content/v2e-output/dvs",
    aps_dir="/content/v2e-output/aps",
    width=346,
    height=260
)

In [ ]:
!ffmpeg -y -framerate 30 -i /content/v2e-output/aps/%04d.png -framerate 30 -i /content/v2e-output/dvs/%04d.png -filter_complex hstack -c:v libx264 -pix_fmt yuv420p /content/v2e-output/aps_dvs.mp4

In [ ]:
!rm -rf '/content/v2e-output/dvs-video-frame_times.txt'
!rm -rf '/content/v2e-output/dvs-video.avi'
!rm -rf '/content/v2e-output/events.h5'
!rm -rf '/content/v2e-output/v2e-args.txt'
!rm -rf '/content/v2e-output/video_orig.avi'
!rm -rf '/content/v2e-output/video_slomo.avi'

In [ ]:
%cd /content/v2e-output
!zip -r /content/{seq}.zip .

import shutil
shutil.copy(
    f"/content/{seq}.zip",
    f"/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/{seq}.zip"
)

In [ ]:
base = f"/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}"
os.environ['SEQ'] = seq
os.environ['BASE'] = base

!unzip "$BASE/$SEQ.zip" -d "$BASE/$SEQ"

# Cycle

In [ ]:
initials = 'seya'

In [ ]:
sequences = ['key_1:2hz', 'key_1hz', 'key_2hz', 'lamp_1:2hz', 'lamp_1hz', 'lamp_2hz',
             'repeater_1:2hz', 'repeater_1hz', 'repeater_2hz', 'soap_1:2hz',
             'soap_1hz', 'soap_2hz', 'white_1:2hz', 'white_1hz', 'white_2hz']

In [ ]:
def create_frames_from_csv(csv_path, output_dir, aps_dir, width=346, height=260):
        os.makedirs(output_dir, exist_ok=True)

        aps_frames = sorted([
            f for f in os.listdir(aps_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        num_frames = len(aps_frames)
        if num_frames == 0:
            raise ValueError(f"No frames found in '{aps_dir}'.")
        print(f"APS frames found: {num_frames}")

        df = pd.read_csv(csv_path)
        t_all = df['t'].to_numpy()
        x_all = df['x'].to_numpy()
        y_all = df['y'].to_numpy()
        p_all = df['p'].to_numpy()

        t_start = t_all[0]
        t_end   = t_all[-1]
        frame_boundaries = np.linspace(t_start, t_end, num_frames + 1)

        event_end_idxs = np.searchsorted(t_all, frame_boundaries[1:], side="right")

        start_idx = 0

        for frame_idx, end_idx in enumerate(tqdm(event_end_idxs, desc="Saving DVS frames")):
            xs = x_all[start_idx:end_idx]
            ys = y_all[start_idx:end_idx]
            ps = p_all[start_idx:end_idx]

            frame = np.zeros((height, width), dtype=np.int16)

            valid = (xs >= 0) & (xs < width) & (ys >= 0) & (ys < height)
            xs = xs[valid]
            ys = ys[valid]
            ps = ps[valid]

            vals = np.where(ps, 1, -1).astype(np.int16, copy=False)
            np.add.at(frame, (ys, xs), vals)

            img = np.full((height, width, 3), 255, dtype=np.uint8)
            img[frame > 0] = (220, 30, 30)
            img[frame < 0] = (30, 30, 200)

            file_name = aps_frames[frame_idx]
            Image.fromarray(img).save(os.path.join(output_dir, file_name))

            start_idx = end_idx

In [ ]:
for seq in tqdm(sequences):
    %cd /content
    !rm -rf '/content/v2e-output'

    os.makedirs('/content/video', exist_ok=True)
    shutil.copy(f'/content/drive/MyDrive/Diploma_project/own_videos/{initials}/{seq}.mp4', '/content/video')

    video_path = f"/content/video/{seq}.mp4"

    if video_path != "" and os.path.isfile(video_path):
        display("The chosen video file: {}".format(video_path))
    else:
        print("The file path is empty or invalid, choose a file")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = frame_count / fps if fps > 0 else 0

    print(f"FPS: {fps}")
    print(f"Frame count: {frame_count}")
    print(f"Duration: {duration_sec:.2f} sec")

    cap.release()

    output_folder = "/content/v2e-output"
    overwrite = True
    unique_output_folder = False
    out_filename = "events.h5"
    davis_output = True
    skip_video_output = False
    dvs_exposure = f"duration {1/fps:.3f}"
    output_mode = "dvs346"
    input_frame_rate = 30
    input_slowmotion_factor = 1
    disable_slomo = False
    timestamp_resolution = 0.001
    auto_timestamp_resolution = True
    slomo_model = "/content/v2e/input/SuperSloMo39.ckpt"
    condition = "Clean"
    thres = 0.2
    sigma = 0.03
    cutoff_hz = 200
    leak_rate_hz = 5.18
    shot_noise_rate_hz = 2.716

    if condition == "Clean":
        thres = 0.2
        sigma = 0.02
        cutoff_hz = 0
        leak_rate_hz = 0
        shot_noise_rate_hz = 0
    elif condition == "Noisy":
        thres = 0.2
        sigma_thres = 0.05
        cutoff_hz = 30
        leak_rate_hz = 0.1
        shot_noise_rate_hz = 5

    v2e_command = ["v2e"]
    v2e_command += ["-i", video_path]
    v2e_command += ["-o", output_folder]

    if overwrite:
        v2e_command.append("--overwrite")

    v2e_command += ["--unique_output_folder", "{}".format(unique_output_folder).lower()]

    if davis_output:
        v2e_command += ["--davis_output"]
    v2e_command += ["--dvs_h5", out_filename]
    v2e_command += ["--dvs_aedat2", "None"]
    v2e_command += ["--dvs_text", "None"]
    v2e_command += ["--no_preview"]

    if skip_video_output:
        v2e_command += ["--skip_video_output"]
    else:
        dvs_exposure = f"duration {1/input_frame_rate:.6f}"
        v2e_command += ["--dvs_exposure", dvs_exposure]

    v2e_command += ["--input_frame_rate", "{}".format(input_frame_rate)]
    v2e_command += ["--input_slowmotion_factor", "{}".format(input_slowmotion_factor)]

    if disable_slomo:
        v2e_command += ["--disable_slomo"]
        v2e_command += ["--auto_timestamp_resolution", "false"]
    else:
        v2e_command += ["--slomo_model", slomo_model]
        if auto_timestamp_resolution:
            v2e_command += ["--auto_timestamp_resolution", "{}".format(auto_timestamp_resolution).lower()]
        else:
            v2e_command += ["--timestamp_resolution", "{}".format(timestamp_resolution)]

    v2e_command += ["--pos_thres", "{}".format(thres)]
    v2e_command += ["--neg_thres", "{}".format(thres)]
    v2e_command += ["--sigma_thres", "{}".format(sigma)]
    v2e_command += ["--cutoff_hz", "{}".format(cutoff_hz)]
    v2e_command += ["--leak_rate_hz", "{}".format(leak_rate_hz)]
    v2e_command += ["--shot_noise_rate_hz", "{}".format(shot_noise_rate_hz)]
    v2e_command += [f"--{output_mode}"]

    final_v2e_command = " ".join(v2e_command)
    print("The Final v2e command:")
    display(final_v2e_command)

    !$final_v2e_command

    # Convert h5 events to csv
    h5_path  = "/content/v2e-output/events.h5"
    csv_path = "/content/v2e-output/events.csv"

    with h5py.File(h5_path, "r") as f:
        events = f["events"][:]

    df = pd.DataFrame(events, columns=["t", "x", "y", "p"])
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")

    # Extract APS frames
    video_path = "/content/v2e-output/video_orig.avi"
    output_dir = "/content/v2e-output/aps"
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_path = os.path.join(output_dir, f"{frame_count + 1:04d}.png")
        cv2.imwrite(frame_path, frame)
        frame_count += 1

    cap.release()
    print("Detected FPS:", fps)
    print("Saved frames:", frame_count)
    print("Output folder:", output_dir)

    # Generate DVS frames
    create_frames_from_csv(
        csv_path="/content/v2e-output/events.csv",
        output_dir="/content/v2e-output/dvs",
        aps_dir="/content/v2e-output/aps",
        width=346,
        height=260
    )

    !ffmpeg -y -framerate 30 -i /content/v2e-output/aps/%04d.png -framerate 30 -i /content/v2e-output/dvs/%04d.png -filter_complex hstack -c:v libx264 -pix_fmt yuv420p /content/v2e-output/aps_dvs.mp4

    !rm -rf '/content/v2e-output/dvs-video-frame_times.txt'
    !rm -rf '/content/v2e-output/dvs-video.avi'
    !rm -rf '/content/v2e-output/events.h5'
    !rm -rf '/content/v2e-output/v2e-args.txt'
    !rm -rf '/content/v2e-output/video_orig.avi'
    !rm -rf '/content/v2e-output/video_slomo.avi'

    %cd /content/v2e-output
    !zip -r /content/{seq}.zip .

    shutil.copy(
        f"/content/{seq}.zip",
        f"/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/{seq}.zip"
    )

    base = f"/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}"
    os.environ['SEQ'] = seq
    os.environ['BASE'] = base
    !unzip "$BASE/$SEQ.zip" -d "$BASE/$SEQ"

    print(f"✓ Done: {seq}")

  0%|          | 0/2 [00:00<?, ?it/s]

/content


'The chosen video file: /content/video/spring_3s.MOV'

FPS: 30.005102908657936
Frame count: 294
Duration: 9.80 sec
The Final v2e command:


'v2e -i /content/video/spring_3s.MOV -o /content/v2e-output --overwrite --unique_output_folder false --davis_output --dvs_h5 events.h5 --dvs_aedat2 None --dvs_text None --no_preview --dvs_exposure duration 0.033333 --input_frame_rate 30 --input_slowmotion_factor 1 --slomo_model /content/v2e/input/SuperSloMo39.ckpt --auto_timestamp_resolution true --pos_thres 0.2 --neg_thres 0.2 --sigma_thres 0.02 --cutoff_hz 0 --leak_rate_hz 0 --shot_noise_rate_hz 0 --dvs346'

...

FPS: 30.004011231448054
Frame count: 374
Duration: 12.46 sec
The Final v2e command:


'v2e -i /content/video/spring_4s.MOV -o /content/v2e-output --overwrite --unique_output_folder false --davis_output --dvs_h5 events.h5 --dvs_aedat2 None --dvs_text None --no_preview --dvs_exposure duration 0.033333 --input_frame_rate 30 --input_slowmotion_factor 1 --slomo_model /content/v2e/input/SuperSloMo39.ckpt --auto_timestamp_resolution true --pos_thres 0.2 --neg_thres 0.2 --sigma_thres 0.02 --cutoff_hz 0 --leak_rate_hz 0 --shot_noise_rate_hz 0 --dvs346'

INFO:v2e:torch device is cuda
INFO:v2e:No module named 'gooey': Gooey GUI builder not available, will use command line arguments.
Install with 'pip install Gooey if you want a no-arg GUI to invoke v2e'. See README
INFO:v2e:name 'Gooey' is not defined: Gooey package GUI not available, using command line arguments. 
You can try to install with "pip install Gooey"
INFO:v2ecore.v2e_utils:using output folder /content/v2e-output
INFO:v2e:output_in_place==False so made output_folder=/content/v2e-output
INFO:v2ecore.v2e_args:
*** arguments:
auto_timestamp_resolution:	True
avi_frame_rate:	30
batch_size:	8
crop:	None
cs_lambda_pixels:	None
cs_tau_p_ms:	None
cutoff_hz:	0.0
ddd_output:	False
disable_slomo:	False
dvs1024:	False
dvs128:	False
dvs240:	False
dvs346:	True
dvs640:	False
dvs_aedat2:	None
dvs_aedat4:	None
dvs_emulator_seed:	0
dvs_exposure:	['duration', '0.033333']
dvs_h5:	events.h5
dvs_params:	None
dvs_text:	None
dvs_vid:	dvs-video.avi
dvs_vid_full_scale:	2
hdr:	False
input:	/content/vide

Saving DVS frames:   0%|          | 0/374 [00:00<?, ?it/s]

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
# if RAM overloaded

def create_frames_from_csv(csv_path, output_dir, aps_dir, width=346, height=260):
    os.makedirs(output_dir, exist_ok=True)

    # Get sorted list of APS frames to match DVS frame count
    aps_frames = sorted([
        f for f in os.listdir(aps_dir)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])
    num_frames = len(aps_frames)
    if num_frames == 0:
        raise ValueError(f"No frames found in '{aps_dir}'.")
    print(f"APS frames found: {num_frames}")

    # Read only first row to get t_start (avoids loading entire CSV)
    first_chunk = pd.read_csv(csv_path, nrows=1)
    t_start = first_chunk['t'].iloc[0]

    # Read only last row to get t_end (avoids loading entire CSV)
    with open(csv_path, 'rb') as f:
        f.seek(-2, 2)
        while f.read(1) != b'\n':
            f.seek(-2, 1)
        last_line = f.readline().decode()
    t_end = float(last_line.split(',')[0])

    # Create evenly spaced time boundaries for each frame
    frame_boundaries = np.linspace(t_start, t_end, num_frames + 1)

    # Pre-allocate accumulation frames (~30 MB total, safe)
    frames = [np.zeros((height, width), dtype=np.int16) for _ in range(num_frames)]

    print("Reading events and building frames (chunked)...")
    chunk_size = 5_000_000  # 5M rows per chunk (~160 MB RAM per iteration)

    for chunk in tqdm(pd.read_csv(csv_path, chunksize=chunk_size),
                      desc="Processing chunks"):
        t = chunk['t'].to_numpy()
        x = chunk['x'].to_numpy()
        y = chunk['y'].to_numpy()
        p = chunk['p'].to_numpy()

        # Assign each event to its corresponding frame index
        frame_idxs = np.searchsorted(frame_boundaries[1:], t, side='right')
        frame_idxs = np.clip(frame_idxs, 0, num_frames - 1)

        # Filter out out-of-bounds pixel coordinates
        valid = (x >= 0) & (x < width) & (y >= 0) & (y < height)
        x = x[valid]
        y = y[valid]
        p = p[valid]
        frame_idxs = frame_idxs[valid]

        # Convert polarity: ON=+1, OFF=-1
        vals = np.where(p, 1, -1).astype(np.int16)

        # Accumulate events into their respective frames
        for fi in np.unique(frame_idxs):
            mask = frame_idxs == fi
            np.add.at(frames[fi], (y[mask], x[mask]), vals[mask])

    # Render and save DVS frames as PNG images
    print("Saving DVS frames...")
    for frame_idx, frame in enumerate(tqdm(frames, desc="Saving DVS frames")):
        img = np.full((height, width, 3), 255, dtype=np.uint8)
        img[frame > 0] = (220, 30, 30)
        img[frame < 0] = (30, 30, 200)
        file_name = aps_frames[frame_idx]
        Image.fromarray(img).save(os.path.join(output_dir, file_name))


for seq in tqdm(sequences):
    %cd /content
    !rm -rf '/content/v2e-output'

    os.makedirs('/content/video', exist_ok=True)
    shutil.copy(f'/content/drive/MyDrive/Diploma_project/own_videos/{initials}/flicker/{seq}.MOV', '/content/video')

    video_path = f"/content/video/{seq}.MOV"

    if video_path != "" and os.path.isfile(video_path):
        display("The chosen video file: {}".format(video_path))
    else:
        print("The file path is empty or invalid, choose a file")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = frame_count / fps if fps > 0 else 0

    print(f"FPS: {fps}")
    print(f"Frame count: {frame_count}")
    print(f"Duration: {duration_sec:.2f} sec")

    cap.release()

    output_folder = "/content/v2e-output"
    overwrite = True
    unique_output_folder = False
    out_filename = "events.h5"
    davis_output = True
    skip_video_output = False
    dvs_exposure = f"duration {1/fps:.3f}"
    output_mode = "dvs346"
    input_frame_rate = 30
    input_slowmotion_factor = 1
    disable_slomo = False
    timestamp_resolution = 0.001
    auto_timestamp_resolution = True
    slomo_model = "/content/v2e/input/SuperSloMo39.ckpt"
    condition = "Clean"
    thres = 0.2
    sigma = 0.03
    cutoff_hz = 200
    leak_rate_hz = 5.18
    shot_noise_rate_hz = 2.716

    if condition == "Clean":
        thres = 0.2
        sigma = 0.02
        cutoff_hz = 0
        leak_rate_hz = 0
        shot_noise_rate_hz = 0
    elif condition == "Noisy":
        thres = 0.2
        sigma_thres = 0.05
        cutoff_hz = 30
        leak_rate_hz = 0.1
        shot_noise_rate_hz = 5

    v2e_command = ["v2e"]
    v2e_command += ["-i", video_path]
    v2e_command += ["-o", output_folder]

    if overwrite:
        v2e_command.append("--overwrite")

    v2e_command += ["--unique_output_folder", "{}".format(unique_output_folder).lower()]

    if davis_output:
        v2e_command += ["--davis_output"]
    v2e_command += ["--dvs_h5", out_filename]
    v2e_command += ["--dvs_aedat2", "None"]
    v2e_command += ["--dvs_text", "None"]
    v2e_command += ["--no_preview"]

    if skip_video_output:
        v2e_command += ["--skip_video_output"]
    else:
        dvs_exposure = f"duration {1/input_frame_rate:.6f}"
        v2e_command += ["--dvs_exposure", dvs_exposure]

    v2e_command += ["--input_frame_rate", "{}".format(input_frame_rate)]
    v2e_command += ["--input_slowmotion_factor", "{}".format(input_slowmotion_factor)]

    if disable_slomo:
        v2e_command += ["--disable_slomo"]
        v2e_command += ["--auto_timestamp_resolution", "false"]
    else:
        v2e_command += ["--slomo_model", slomo_model]
        if auto_timestamp_resolution:
            v2e_command += ["--auto_timestamp_resolution", "{}".format(auto_timestamp_resolution).lower()]
        else:
            v2e_command += ["--timestamp_resolution", "{}".format(timestamp_resolution)]

    v2e_command += ["--pos_thres", "{}".format(thres)]
    v2e_command += ["--neg_thres", "{}".format(thres)]
    v2e_command += ["--sigma_thres", "{}".format(sigma)]
    v2e_command += ["--cutoff_hz", "{}".format(cutoff_hz)]
    v2e_command += ["--leak_rate_hz", "{}".format(leak_rate_hz)]
    v2e_command += ["--shot_noise_rate_hz", "{}".format(shot_noise_rate_hz)]
    v2e_command += [f"--{output_mode}"]

    final_v2e_command = " ".join(v2e_command)
    print("The Final v2e command:")
    display(final_v2e_command)

    !$final_v2e_command

    # Convert h5 events to csv (chunked, memory-safe)
    h5_path  = "/content/v2e-output/events.h5"
    csv_path = "/content/v2e-output/events.csv"
    chunk_size = 5_000_000  # 5M rows per chunk (~160 MB RAM)

    with h5py.File(h5_path, "r") as f:
        total = f["events"].shape[0]
        print(f"Total events: {total:,}")
        first = True
        for start in range(0, total, chunk_size):
            chunk = f["events"][start:start + chunk_size]
            df = pd.DataFrame(chunk, columns=["t", "x", "y", "p"])
            df.to_csv(csv_path, mode='w' if first else 'a',
                      index=False, header=first)
            first = False
            print(f"  Written {min(start + chunk_size, total):,}/{total:,}")

    print(f"Saved: {csv_path}")

    # Extract APS frames
    video_path = "/content/v2e-output/video_orig.avi"
    output_dir = "/content/v2e-output/aps"
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_path = os.path.join(output_dir, f"{frame_count + 1:04d}.png")
        cv2.imwrite(frame_path, frame)
        frame_count += 1

    cap.release()
    print("Detected FPS:", fps)
    print("Saved frames:", frame_count)
    print("Output folder:", output_dir)

    # Generate DVS frames (chunked, memory-safe)
    create_frames_from_csv(
        csv_path="/content/v2e-output/events.csv",
        output_dir="/content/v2e-output/dvs",
        aps_dir="/content/v2e-output/aps",
        width=346,
        height=260
    )

    !ffmpeg -y -framerate 30 -i /content/v2e-output/aps/%04d.png -framerate 30 -i /content/v2e-output/dvs/%04d.png -filter_complex hstack -c:v libx264 -pix_fmt yuv420p /content/v2e-output/aps_dvs.mp4

    !rm -rf '/content/v2e-output/dvs-video-frame_times.txt'
    !rm -rf '/content/v2e-output/dvs-video.avi'
    !rm -rf '/content/v2e-output/events.h5'
    !rm -rf '/content/v2e-output/v2e-args.txt'
    !rm -rf '/content/v2e-output/video_orig.avi'
    !rm -rf '/content/v2e-output/video_slomo.avi'

    %cd /content/v2e-output
    !zip -r /content/{seq}.zip .

    shutil.copy(
        f"/content/{seq}.zip",
        f"/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}.zip"
    )

    base = f"/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker"
    os.environ['SEQ'] = seq
    os.environ['BASE'] = base
    !unzip "$BASE/$SEQ.zip" -d "$BASE/$SEQ"

    print(f"✓ Done: {seq}")

  0%|          | 0/15 [00:00<?, ?it/s]

/content


'The chosen video file: /content/video/key_1:2hz.MOV'

FPS: 29.972375690607734
Frame count: 217
Duration: 7.24 sec
The Final v2e command:


'v2e -i /content/video/key_1:2hz.MOV -o /content/v2e-output --overwrite --unique_output_folder false --davis_output --dvs_h5 events.h5 --dvs_aedat2 None --dvs_text None --no_preview --dvs_exposure duration 0.033333 --input_frame_rate 30 --input_slowmotion_factor 1 --slomo_model /content/v2e/input/SuperSloMo39.ckpt --auto_timestamp_resolution true --pos_thres 0.2 --neg_thres 0.2 --sigma_thres 0.02 --cutoff_hz 0 --leak_rate_hz 0 --shot_noise_rate_hz 0 --dvs346'

...

INFO:v2e:torch device is cuda
INFO:v2e:No module named 'gooey': Gooey GUI builder not available, will use command line arguments.
Install with 'pip install Gooey if you want a no-arg GUI to invoke v2e'. See README
INFO:v2e:name 'Gooey' is not defined: Gooey package GUI not available, using command line arguments. 
You can try to install with "pip install Gooey"
INFO:v2ecore.v2e_utils:using output folder /content/v2e-output
INFO:v2e:output_in_place==False so made output_folder=/content/v2e-output
INFO:v2ecore.v2e_args:
*** arguments:
auto_timestamp_resolution:	True
avi_frame_rate:	30
batch_size:	8
crop:	None
cs_lambda_pixels:	None
cs_tau_p_ms:	None
cutoff_hz:	0.0
ddd_output:	False
disable_slomo:	False
dvs1024:	False
dvs128:	False
dvs240:	False
dvs346:	True
dvs640:	False
dvs_aedat2:	None
dvs_aedat4:	None
dvs_emulator_seed:	0
dvs_exposure:	['duration', '0.033333']
dvs_h5:	events.h5
dvs_params:	None
dvs_text:	None
dvs_vid:	dvs-video.avi
dvs_vid_full_scale:	2
hdr:	False
input:	/content/vide

Processing chunks: 0it [00:00, ?it/s]

Saving DVS frames...


Saving DVS frames:   0%|          | 0/201 [00:00<?, ?it/s]

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

# Process GT

In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

from tqdm.notebook import tqdm

Mounted at /content/drive


In [ ]:
seq = 'tag'
initials = 'seya'


gt_path = f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/clean/{seq}/gt.txt'

df = pd.read_csv(
    gt_path,
    header=None,
    names=["frame", "id", "x", "y", "w", "h", "conf", "class", "visibility"]
)
df

,frame,id,x,y,w,h,conf,class,visibility
0,1,1,231.85,114.81,16.30,18.89,1,1,1.0
1,2,1,213.44,119.66,16.53,18.82,1,1,1.0
2,3,1,193.98,122.17,13.88,18.48,1,1,1.0
3,4,1,174.92,120.41,14.88,17.58,1,1,1.0
4,5,1,156.92,115.42,14.70,18.18,1,1,1.0
...,...,...,...,...,...,...,...,...,...
309,310,1,163.70,115.04,12.77,18.21,1,1,1.0
310,311,1,152.35,110.82,13.04,18.47,1,1,1.0
311,312,1,142.56,105.84,13.65,19.03,1,1,1.0
312,313,1,133.58,100.47,14.09,18.84,1,1,1.0


In [ ]:
df_new = df[["x", "y", "w", "h"]]
df_new

,x,y,w,h
0,231.85,114.81,16.30,18.89
1,213.44,119.66,16.53,18.82
2,193.98,122.17,13.88,18.48
3,174.92,120.41,14.88,17.58
4,156.92,115.42,14.70,18.18
...,...,...,...,...
309,163.70,115.04,12.77,18.21
310,152.35,110.82,13.04,18.47
311,142.56,105.84,13.65,19.03
312,133.58,100.47,14.09,18.84


In [ ]:
new_gt_path = f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/clean/{seq}/groundtruth_rect.txt'
df_new.to_csv(new_gt_path, sep=",", header=False, index=False)

In [ ]:
initials = 'seya'
sequences = ['key_1:2hz', 'key_1hz', 'key_2hz', 'lamp_1:2hz', 'lamp_1hz', 'lamp_2hz',
             'repeater_1:2hz', 'repeater_1hz', 'repeater_2hz', 'soap_1:2hz',
             'soap_1hz', 'soap_2hz', 'white_1:2hz', 'white_1hz', 'white_2hz',
             'cleaner_1:2hz', 'cleaner_1hz', 'cleaner_2hz', 'knife_1:2hz', 'knife_1hz', 'knife_2hz',
             'longus_1:2hz', 'longus_1hz', 'longus_2hz', 'nonstop_1:2hz',
             'nonstop_1hz', 'nonstop_2hz', 'rexona_1:2hz', 'rexona_1hz', 'rexona_2hz']

for seq in tqdm(sequences):
    gt_path = f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/gt.txt'

    df = pd.read_csv(
        gt_path,
        header=None,
        names=["frame", "id", "x", "y", "w", "h", "conf", "class", "visibility"]
    )
    df_new = df[["x", "y", "w", "h"]]

    new_gt_path = f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/groundtruth_rect.txt'
    df_new.to_csv(new_gt_path, sep=",", header=False, index=False)


  0%|          | 0/30 [00:00<?, ?it/s]

# Video aps_dvs_gpt with gt

In [ ]:
import os
import shutil
import cv2
import numpy as np
from tqdm.notebook import tqdm

In [ ]:
def load_bboxes(path):
    bboxes = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                d = ',' if ',' in line else '\t'
                bboxes.append(list(map(float, line.split(d))))
    return bboxes

In [ ]:
initials = 'seya'
sequences = ['key_1:2hz', 'key_1hz', 'key_2hz', 'lamp_1:2hz', 'lamp_1hz', 'lamp_2hz',
             'repeater_1:2hz', 'repeater_1hz', 'repeater_2hz', 'soap_1:2hz',
             'soap_1hz', 'soap_2hz', 'white_1:2hz', 'white_1hz', 'white_2hz',
             'cleaner_1:2hz', 'cleaner_1hz', 'cleaner_2hz', 'knife_1:2hz', 'knife_1hz', 'knife_2hz',
             'longus_1:2hz', 'longus_1hz', 'longus_2hz', 'nonstop_1:2hz',
             'nonstop_1hz', 'nonstop_2hz', 'rexona_1:2hz', 'rexona_1hz', 'rexona_2hz']


for seq in tqdm(sequences):
    base = f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}'
    tmp = f'/tmp/{seq}'
    os.makedirs(tmp, exist_ok=True)

    gt_boxes   = load_bboxes(f'{base}/groundtruth_rect.txt')
    aps_frames = sorted(os.listdir(f'{base}/aps'))
    dvs_frames = sorted(os.listdir(f'{base}/dvs'))
    gtp_frames = sorted(os.listdir(f'{base}/inter1_stack_3008'))
    n = min(len(aps_frames), len(dvs_frames), len(gtp_frames), len(gt_boxes))

    for i in tqdm(range(n)):
        aps_img = cv2.imread(f'{base}/aps/{aps_frames[i]}')
        dvs_img = cv2.imread(f'{base}/dvs/{dvs_frames[i]}')
        gtp_img = cv2.imread(f'{base}/inter1_stack_3008/{gtp_frames[i]}')

        labels = ['APS', 'GTP', 'DVS']
        for fr, label in zip([aps_img, gtp_img, dvs_img], labels):
            # draw GT bounding box in green
            x, y, w, h = [int(v) for v in gt_boxes[i]]
            if any([x, y, w, h]):
                cv2.rectangle(fr, (x, y), (x+w, y+h), (0, 255, 0), 1)

            # window label in top-left corner
            cv2.putText(fr, label, (10, 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

            # frame number at the bottom
            cv2.putText(fr, f'Frame: {i+1}', (10, fr.shape[0] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.27, (255, 255, 255), 1)

        # stack frames horizontally and save
        cv2.imwrite(f'{tmp}/{i:04d}.png', np.hstack([aps_img, gtp_img, dvs_img]))

    # encode frames to mp4
    !ffmpeg -y -framerate 30 -i {tmp}/%04d.png -c:v libx264 -pix_fmt yuv420p {base}/aps_dvs_gtp.mp4
    shutil.rmtree(tmp)

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/217 [00:00<?, ?it/s]

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

  0%|          | 0/221 [00:00<?, ?it/s]

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

...

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

  0%|          | 0/224 [00:00<?, ?it/s]

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

  0%|          | 0/222 [00:00<?, ?it/s]

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
from google.colab import files

for seq in sequences:
    path = f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/flicker/{seq}/aps_dvs_gtp.mp4'
    shutil.copy(path, f'/content/{seq}.mp4')
    files.download(f'/content/{seq}.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Complex video

In [ ]:
import cv2
import numpy as np
import os
import shutil
from tqdm import tqdm

# Directories
base = '/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/clean/bracket'
dvs_dir    = f'{base}/dvs'
gtp_dirs   = [f'{base}/augmented/gtp_{i}' for i in range(1, 11)]
gt_path    = f'{base}/groundtruth_rect.txt'
output_mp4 = f'{base}/augmented/comparison.mp4'
tmp_dir    = '/tmp/comparison'

# Load ground truth bounding boxes (x,y,w,h)
gt_boxes = []
with open(gt_path, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        vals = line.replace(',', ' ').split()
        gt_boxes.append([int(float(v)) for v in vals[:4]])

def draw_box(img, box, color):
    x, y, w, h = box
    cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
    return img

def add_label(img, text):
    cv2.putText(img, text, (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    cv2.putText(img, text, (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1)
    return img

def read_img(path):
    img = cv2.imread(path)
    if img is None:
        return None
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    return img

# Get sorted frame lists
dvs_frames = sorted(os.listdir(dvs_dir))
gtp_frames = [sorted(os.listdir(d)) for d in gtp_dirs]
n = min(len(dvs_frames), *[len(f) for f in gtp_frames], len(gt_boxes))

os.makedirs(tmp_dir, exist_ok=True)

for i in tqdm(range(n)):
    box = gt_boxes[i]

    # Read DVS, draw box and label, then scale to 2x2
    dvs = read_img(f'{dvs_dir}/{dvs_frames[i]}')
    H, W = dvs.shape[:2]
    draw_box(dvs, box, color=(0, 0, 255))
    add_label(dvs, 'DVS')
    dvs_big = cv2.resize(dvs, (W * 2, H * 2))

    # Read GTP frames, draw box and label on original size
    gtps = []
    for j, d in enumerate(gtp_dirs):
        img = read_img(f'{d}/{gtp_frames[j][i]}')
        if img is None:
            img = np.zeros((H, W, 3), dtype=np.uint8)
        img = cv2.resize(img, (W, H))
        draw_box(img, box, color=(0, 255, 0))
        add_label(img, f'gtp_{j+1}')
        gtps.append(img)

    # Build grid
    empty = np.zeros((H, W, 3), dtype=np.uint8)

    right_top = np.vstack([
        np.hstack([gtps[0], gtps[1]]),
        np.hstack([gtps[2], gtps[3]])
    ])
    row_top = np.hstack([dvs_big, right_top])
    row_mid = np.hstack([gtps[4], gtps[5], gtps[6], gtps[7]])
    row_bot = np.hstack([gtps[8], gtps[9], empty, empty])

    frame = np.vstack([row_top, row_mid, row_bot])
    cv2.imwrite(f'{tmp_dir}/{i:04d}.png', frame)

# Render video
os.system(f'ffmpeg -y -framerate 30 -i {tmp_dir}/%04d.png -c:v libx264 -pix_fmt yuv420p {output_mp4}')
shutil.rmtree(tmp_dir)
print(f'Done! Video saved: {output_mp4}')

100%|██████████| 270/270 [16:44<00:00,  3.72s/it]


Done! Video saved: /content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/clean/bracket/augmented/comparison.mp4


In [ ]:
import cv2
import numpy as np
import os
import shutil
from tqdm import tqdm

def draw_box(img, box, color):
    x, y, w, h = box
    cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
    return img

def add_label(img, text):
    cv2.putText(img, text, (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    cv2.putText(img, text, (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1)
    return img

def read_img(path):
    img = cv2.imread(path)
    if img is None:
        return None
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    return img

initials = 'seya'
sequences = ['clamp', 'lighter', 'ruler', 'spinner', 'tag', 'traumel']

for seq in sequences:
    print(f'\nProcessing: {seq}')
    base       = f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/clean/{seq}'
    dvs_dir    = f'{base}/dvs'
    gtp_dirs   = [f'{base}/augmented/gtp_{i}' for i in range(1, 11)]
    gt_path    = f'{base}/groundtruth_rect.txt'
    output_mp4 = f'{base}/augmented/comparison.mp4'
    tmp_dir    = f'/tmp/comparison_{seq}'

    # Load ground truth bounding boxes (x,y,w,h)
    gt_boxes = []
    with open(gt_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            vals = line.replace(',', ' ').split()
            gt_boxes.append([int(float(v)) for v in vals[:4]])

    # Get sorted frame lists
    dvs_frames = sorted(os.listdir(dvs_dir))
    gtp_frames = [sorted(os.listdir(d)) for d in gtp_dirs]
    n = min(len(dvs_frames), *[len(f) for f in gtp_frames], len(gt_boxes))

    os.makedirs(tmp_dir, exist_ok=True)

    for i in tqdm(range(n), desc=seq):
        box = gt_boxes[i]

        # Read DVS, draw box and label, then scale to 2x2
        dvs = read_img(f'{dvs_dir}/{dvs_frames[i]}')
        if dvs is None:
            print(f'Warning: skipping frame {i}, DVS not readable: {dvs_frames[i]}')
            continue
        H, W = dvs.shape[:2]
        draw_box(dvs, box, color=(0, 0, 255))
        add_label(dvs, 'DVS')
        dvs_big = cv2.resize(dvs, (W * 2, H * 2))

        # Read GTP frames, draw box and label on original size
        gtps = []
        for j, d in enumerate(gtp_dirs):
            img = read_img(f'{d}/{gtp_frames[j][i]}')
            if img is None:
                img = np.zeros((H, W, 3), dtype=np.uint8)
            img = cv2.resize(img, (W, H))
            draw_box(img, box, color=(0, 255, 0))
            add_label(img, f'gtp_{j+1}')
            gtps.append(img)

        # Build grid
        empty = np.zeros((H, W, 3), dtype=np.uint8)

        right_top = np.vstack([
            np.hstack([gtps[0], gtps[1]]),
            np.hstack([gtps[2], gtps[3]])
        ])
        row_top = np.hstack([dvs_big, right_top])
        row_mid = np.hstack([gtps[4], gtps[5], gtps[6], gtps[7]])
        row_bot = np.hstack([gtps[8], gtps[9], empty, empty])

        frame = np.vstack([row_top, row_mid, row_bot])
        cv2.imwrite(f'{tmp_dir}/{i:04d}.png', frame)

    # Render video
    os.system(f'ffmpeg -y -framerate 30 -i {tmp_dir}/%04d.png -c:v libx264 -pix_fmt yuv420p {output_mp4}')
    shutil.rmtree(tmp_dir)
    print(f'Done: {output_mp4}')


Processing: clamp


clamp: 100%|██████████| 378/378 [28:45<00:00,  4.56s/it]


Done: /content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/clean/clamp/augmented/comparison.mp4

Processing: lighter


lighter: 100%|██████████| 326/326 [23:36<00:00,  4.35s/it]


Done: /content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/clean/lighter/augmented/comparison.mp4

Processing: ruler


...

Done: /content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/clean/spinner/augmented/comparison.mp4

Processing: tag


tag: 100%|██████████| 314/314 [26:14<00:00,  5.01s/it]


Done: /content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/clean/tag/augmented/comparison.mp4

Processing: traumel


traumel: 100%|██████████| 312/312 [25:09<00:00,  4.84s/it]


Done: /content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/clean/traumel/augmented/comparison.mp4
